# Prolog Program Explainer using LangChain

This notebook creates a tool that provides natural language explanations of Prolog programs using LangChain.

In [5]:
# Import necessary libraries
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import StrOutputParser
from langchain_ollama import OllamaLLM
import shutil

# Load environment variables (especially API keys)
# load_dotenv()

In [6]:
ollama_exists = shutil.which('ollama') is not None

if not ollama_exists:
    # Install Ollama (if not already installed)
    !curl -fsSL https://ollama.com/install.sh | sh

    # Start the Ollama server
    !ollama serve

In [7]:
# Define the output structure with a Pydantic model
class PrologExplanation(BaseModel):
    summary: str = Field(description="A brief summary of what the Prolog program does")
    primitive: List[dict] = Field(description="List of dependent primitives and their descriptions")
    explanation: str = Field(description="How the program logic works")
    
def explain_prolog(llm, prompt_template, prolog_code, variables=['prolog_code']):
    """Generate a natural language explanation of a Prolog program.
    
    Args:
        llm: The language model to use
        prompt: The prompt template to use
        prolog_code (str): The Prolog code to explain
        
    Returns:
        PrologExplanation: Structured explanation of the Prolog program
    """
    # Initialize the output parser
    parser = PydanticOutputParser(pydantic_object=PrologExplanation)
    
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=variables,
        partial_variables={"JSON_format": parser.get_format_instructions()}
    )
    
    # Get raw string response from the LLM
    chain = prompt | llm | StrOutputParser()
    raw_result = chain.invoke({"prolog_code": prolog_code})
    
    print(f"===== Generated Raw Text =====\n{raw_result}")
    
    # Parse the raw string into a structured PrologExplanation object
    try:
        # Try to parse the output directly
        parsed_result = parser.parse(raw_result)
        
        # print(f"\n===== Parsed Prolog Explanation =====\n{parsed_result}")
        print(f"\n===== Parsed Text =====")

        print(f"Summary: {parsed_result.summary}\n")
        print("Primitives:")
        for pred in parsed_result.primitive:
            print(f"- {pred.get('name')}: {pred.get('description')}")
        print(f"\nExplanation: {parsed_result.explanation}\n")
        return parsed_result
    except Exception as e:
        # If direct parsing fails, attempt to extract structured data from text
        print(f"Parsing error: {e}")
        print("Returning raw output instead. Check format of LLM response.")
        return raw_result

In [ ]:
# Llama 3.2 3B
llm = OllamaLLM(
    model = 'llama3.2',
    temperature = 0
)

In [9]:
explanation_template = """
You are an expert Prolog interpreter who explains Prolog code to beginners.

I. Analyze the following Prolog program which is divided into three sections:
1. Target rules: The main predicates that define the goal of the program
2. Primitives: Supporting predicates that help implement the target rules
3. Biases: Input and output signature directions for the primitives

{prolog_code}

II. Your response MUST follow the below JSON format EXACTLY with no additional text:

{JSON_format}

YOUR EXPLANATION MUST:

In "summary": Provide a 1-2 sentence overview of what the entire Prolog program accomplishes.
In "primitive": List each primitive from Target rules as dictionaries with "name" and "description" keys.
In "explanation": Explain how the Target rules use the Primitives to achieve the program's purpose.

IMPORTANT FORMATTING RULES:

Return ONLY valid parseable JSON with the exact field names shown above.
Ensure all strings are properly escaped for valid JSON.
DO NOT return JSON schema information or "properties" fields.
Do not include markdown formatting symbols, code blocks, or explanatory text outside the JSON structure.
"""

## Example Usage

In [ ]:
# Example Prolog code - Family relationships
family_prolog = """
%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
parent(X, Y) :- father(X, Y).
parent(X, Y) :- mother(X, Y).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
# direction(grandparent,(in,out)).
direction(parent,(in,out)). 
direction(father,(in,out)). 
direction(mother,(in,out)). 
"""

explanation = explain_prolog(llm, explanation_template, family_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program defines a relationship between grandparents and their descendants, using parentage relationships to establish lineage.", "primitive": [{"name": "parent", "description": "Determines if one entity is a parent of another"}, {"name": "father", "description": "Establishes the father-child relationship"}, {"name": "mother", "description": "Establishes the mother-child relationship"}], "explanation": "The program uses the parent predicate to determine if one entity is a parent of another, and then applies this relationship to establish the grandparent-grandchild relationship through the grandparent predicate."}

===== Parsed Text =====
Summary: This Prolog program defines a relationship between grandparents and their descendants, using parentage relationships to establish lineage.

Primitives:
- parent: Determines if one entity is a parent of another
- father: Establishes the father-child relationship
- mother: Establishes the mo

## Custom Prolog Code Explanation

Without code comments

In [ ]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

inv_ho_0(A,B):- same_circuit(A,B),subpath(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(subpath(B,A)).

partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
all(P, X, L):- 
    findall(H, call(P, X, H), L).

has_branches(X) :- is_connected(X, Y), is_connected(X, Z), Y \\== Z.

subpath(X, X).
subpath(X, Y) :- is_connected(X, Z),  not(has_branches(X)), subpath(Z, Y).
not_subpath(X, Y) :- not(subpath(X, Y)).

same_circuit(X, Y) :- gate(X), gate(Y), N is X // 100, M is Y // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
direction(partition,(in,out,out)). 
direction(subpath,(in,out)).
direction(not_subpath,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program is designed to partition a circuit into two subpaths based on whether they share a common gate and are connected to different branches.", "primitive": [{"name": "all", "description": "Finds all solutions for a given predicate"}, {"name": "has_branches", "description": "Checks if a node has multiple connections"}, {"name": "is_connected", "description": "Checks if two nodes are connected through the same path"}, {"name": "not_subpath", "description": "Checks if a subpath does not exist between two nodes"}, {"name": "same_circuit", "description": "Checks if two nodes share the same gate and have the same index in the circuit"}, {"name": "subpath", "description": "Finds a subpath between two nodes"}, {"name": "partition", "description": "Partitions a circuit into two subpaths based on common gates and connections"}], "explanation": "The program uses the primitives to find all solutions for the target rules. The partition rule

With code comments

In [ ]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

% Gates affected by a test and those not affected
inv_ho_1(A,B):- same_circuit(A,B),not(subpath(B,A)).
inv_ho_0(A,B):- same_circuit(A,B),subpath(B,A).

% Given Gate A find gates on the same branch as A and gates on other branches
partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
% HO predicate
all(P, X, L):- 
    findall(H, call(P, X, H), L).

% Are multiple gates connected to gate X?
has_branches(X) :- is_connected(X, Y), is_connected(X, Z), Y \\== Z.

% Is gate X to gate Y a path (no branches)?
subpath(X, X).
subpath(X, Y) :- is_connected(X, Z),  not(has_branches(X)), subpath(Z, Y).
not_subpath(X, Y) :- not(subpath(X, Y)).

% Does gate X share a subpath with gate Y (Y precedes X if X \\== Y)?
same_circuit(X, Y) :- gate(X), gate(Y), N is X // 100, M is Y // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
direction(partition,(in,out,out)). 
direction(subpath,(in,out)).
direction(not_subpath,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program determines whether two gates share a subpath and partitions gates into branches based on this relationship.", "primitive": [{"name": "all", "description": "Finds all solutions to a predicate"}, {"name": "has_branches", "description": "Checks if multiple gates are connected to the same gate"}, {"name": "is_connected", "description": "Checks if two gates are connected by a path"}, {"name": "same_circuit", "description": "Checks if two gates share the same branch in a circuit"}, {"name": "subpath", "description": "Checks if one gate is on the subpath of another gate"}, {"name": "not_subpath", "description": "Checks if two gates do not share a subpath"}], "explanation": "The program uses the primitives to determine whether two gates are connected by a path (subpath) or not. It then uses these connections to partition gates into branches, with gates on the same branch being grouped together and those on different branches being